In [ ]:
library(SeuratObject)
library(Seurat)
library(SeuratDisk)
#library(presto)
library(Matrix)
library(reticulate)
library(dplyr)

In [ ]:
sc <-readRDS('data/spatial/20230911_tonsil_atlas_rna_seurat_obj.rds')
st <-readRDS('data/spatial/20220527_tonsil_atlas_spatial_seurat_obj.rds')

In [ ]:
sc[["percent.mt"]] <- PercentageFeatureSet(sc, pattern = "^MT-")
sc <- subset(sc, subset = nFeature_RNA > 500 & nFeature_RNA < 6000 & 
                                  nCount_RNA > 1000 & nCount_RNA < 30000 & percent.mt <10)

In [ ]:
counts <- sc[["RNA"]]@counts
meta <- sc@meta.data[, c("barcode", "donor_id","annotation_level_1")]
sc <- CreateSeuratObject(counts =counts , meta.data = meta)

counts <- st[["Spatial"]]@counts
meta <- st@meta.data[, c("barcode", "donor_id")]
rownames(meta) <- meta$barcode
images <- st@images
st <- CreateSeuratObject(counts = counts, meta.data = meta,assay = "Spatial")
st@images <- images

In [ ]:
donor = c('BCLL-8-T','BCLL-9-T','BCLL-10-T','BCLL-11-T','BCLL-12-T','BCLL-13-T')
sc <- subset(sc, subset = donor_id %in% donor)
sc

In [ ]:
url <- "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/locus_types/gene_with_protein_product.txt"
hgnc <- read.delim(url, sep = "\t", header = TRUE, stringsAsFactors = FALSE)
pc_genes <- hgnc$symbol
keep_sc <- intersect(rownames(sc), pc_genes)
keep_sc <- keep_sc[!grepl("^MT-", keep_sc)]

keep_st <- intersect(rownames(st), pc_genes)
keep_st <- keep_st[!grepl("^MT-", keep_st)]

keep_common <- intersect(keep_sc, keep_st)
sc <- DietSeurat(sc, features = keep_common)
st <- DietSeurat(st, features = keep_common)

In [ ]:
sc

In [ ]:
st

In [ ]:
saveRDS(sc, file = "data/spatial/processed_data/scRNA.rds")
saveRDS(st, file = "data/spatial/processed_data/spatial.rds")